# F0: Validación de Excel originales.

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la Universitat Jaume I**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |
| **Versión** | AU_UJI Dinámico (V2) |
| **Fase** | 0 — Configuración |

---

## 🎯 ¿Qué hace este notebook?

Este notebook es el **paso de validación previo** a cualquier fase del proyecto. Verifica que los 2 ficheros Excel originales (proporcionados por OPP / UADTI de la Universitat Jaume I) cumplen el contrato esperado **antes** de que F1 empiece a procesarlos.

Detectar problemas estructurales aquí evita horas de depuración posterior en F1-F7.

## 📋 Niveles de validación

El validador comprueba **5 niveles**, agrupados por criticidad:

| Nivel | Tipo | Qué valida |
|---|---|---|
| **N1** | 🔴 bloqueante | Existencia de los 2 ficheros, tamaño > 0 y legibilidad |
| **N2** | 🔴 bloqueante | Estructura: 8 hojas en el Excel principal + Hoja1 en preinscripción |
| **N3** | 🔴 bloqueante | Columnas obligatorias presentes en cada hoja (case-insensitive) |
| **N4** | 🟡 aviso | Tipos de datos coherentes con el contrato |
| **N5** | 🟡 aviso | Volumen razonable de filas y cruces de IDs entre hojas |

Los niveles **bloqueantes** (N1-N3) impiden continuar con F1 si fallan. Los **avisos** (N4-N5) no bloquean pero conviene revisarlos.

## ⚠️ Requisitos

- Los 2 ficheros Excel deben estar en `data/00_raw/`:
  - `datos_proyecto_sin_preinscrip.xlsx` (8 hojas)
  - `preinscripcion_si.xlsx` (1 hoja)
- El contrato de validación está definido en `src/validacion/contrato_excel.py`

## 📦 Genera

- **Salida en consola**: resumen por niveles con detalles de cada validación
- **HTML autocontenido**: `docs/html/fase0/validacion_excel.html` con informe visual completo (cards por nivel, detalles desplegables, cartel institucional para futuros investigadores)

## 🔄 Flujo

```
Notebook → src.validacion.validador_excel.ejecutar_validacion_completa()
        → src.validacion.generar_html_validacion.generar_html_validacion()
        → docs/html/fase0/validacion_excel.html
```

## ➡️ Siguiente paso

- Si `Todos OK = True` → continuar con `f1_m01_reportes_raw.ipynb` (Fase 1)
- Si hay fallos bloqueantes → revisar el HTML, corregir los Excel y volver a ejecutar

---


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
#
# Detecta entorno (Colab/Local), configura ROOT buscando 'src/',
# e importa las funciones del paquete src.validacion.
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# --- Detectar entorno ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI_v2/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto (paquete validacion) ---
from src.validacion.validador_excel import ejecutar_validacion_completa
from src.validacion.generar_html_validacion import generar_html_validacion
from src.validacion.contrato_excel import CONTRATO_EXCEL, hojas_esperadas

# --- Mostrar información ---
print('=' * 60)
print('VALIDACIÓN DE EXCEL — Fase 0')
print('=' * 60)
print(f'Fecha de ejecución: {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}')
print(f'ROOT del proyecto:  {ROOT}')
print(f'Hojas en el contrato: {len(CONTRATO_EXCEL)}')
print(f'  • Excel principal:      {len(hojas_esperadas("principal"))} hojas')
print(f'  • Excel preinscripción: {len(hojas_esperadas("preinscripcion"))} hoja')


VALIDACIÓN DE EXCEL — Fase 0
Fecha de ejecución: 04/05/2026 14:37:20
ROOT del proyecto:  c:\FF\AU_UJI_v2
Hojas en el contrato: 9
  • Excel principal:      8 hojas
  • Excel preinscripción: 1 hoja


In [2]:
# ============================================================================
# CELDA 2: EJECUTAR LAS 5 VALIDACIONES (N1-N5)
# ============================================================================
#
# Lanza la cadena completa de validaciones contra los 2 Excel originales.
# Tiempo aproximado: 2-3 minutos (N5 carga columnas grandes para validar
# cruces de IDs entre hojas).
#
# Si N1, N2 o N3 fallan → bloqueante_fallido = True (NO continuar a F1).
# Si solo N4-N5 dan avisos → continuar pero revisar el HTML.
# ============================================================================

print('=' * 60)
print('EJECUTANDO VALIDACIONES (N1 → N5)')
print('=' * 60)
print('Tiempo estimado: 2-3 minutos')
print()

# verbose=True imprime progreso por consola en tiempo real
resultado = ejecutar_validacion_completa(verbose=True)

print()
print('=' * 60)
print('RESUMEN POR NIVELES')
print('=' * 60)
print(resultado['resumen_corto'])
print()
print(f'Bloqueante fallido: {resultado["bloqueante_fallido"]}')
print(f'Todos OK:           {resultado["todos_ok"]}')


EJECUTANDO VALIDACIONES (N1 → N5)
Tiempo estimado: 2-3 minutos

  → Ejecutando N1...
    ✅ N1 OK
  → Ejecutando N2...
    ✅ N2 OK
  → Ejecutando N3...
    ✅ N3 OK
  → Ejecutando N4...
    ✅ N4 OK
  → Ejecutando N5...
    ✅ N5 OK

RESUMEN POR NIVELES
  ✅ N1 (Existencia de ficheros): 2 OK, 0 avisos/errores
  ✅ N2 (Estructura de hojas): 2 OK, 0 avisos/errores
  ✅ N3 (Columnas obligatorias): 9 OK, 0 avisos/errores
  ✅ N4 (Tipos de datos): 9 OK, 0 avisos/errores
  ✅ N5 (Volumen y cruces de IDs): 12 OK, 0 avisos/errores

Bloqueante fallido: False
Todos OK:           True


In [3]:
# ============================================================================
# CELDA 3: GENERAR EL HTML DEL INFORME
# ============================================================================
#
# Convierte el dict de resultados en un informe HTML autocontenido (CSS
# embebido, sin dependencias externas).
#
# Se guarda en: docs/html/fase0/validacion_excel.html
# El HTML incluye:
#   • Resumen visual con 5 cards (una por nivel)
#   • Detalles desplegables con todos los mensajes
#   • Cartel institucional OPP / UADTI para futuros investigadores
# ============================================================================

print('=' * 60)
print('GENERANDO INFORME HTML')
print('=' * 60)

ruta_html = generar_html_validacion(resultado)

print(f'\n  ✓ HTML generado en:')
print(f'    {ruta_html}')
print()
print('  Para abrirlo:')
print(f'    • Doble clic en el fichero, o')
print(f'    • Navegador → File → Open → {ruta_html.name}')


GENERANDO INFORME HTML

  ✓ HTML generado en:
    c:\FF\AU_UJI_v2\docs\html\fase0\validacion_excel.html

  Para abrirlo:
    • Doble clic en el fichero, o
    • Navegador → File → Open → validacion_excel.html


In [4]:
# ============================================================================
# CELDA 4: CONCLUSIÓN Y SIGUIENTE PASO
# ============================================================================
#
# Imprime un mensaje final claro indicando si se puede continuar con F1
# o si hay que corregir problemas antes.
# ============================================================================

print('=' * 60)
print('CONCLUSIÓN')
print('=' * 60)

if resultado['bloqueante_fallido']:
    print('\n  ✗ VALIDACIÓN FALLIDA — uno o más niveles bloqueantes (N1-N3)')
    print('    NO han pasado. Revisar el HTML para ver los detalles.')
    print()
    print('  Siguiente paso:')
    print('    1. Abrir docs/html/fase0/validacion_excel.html')
    print('    2. Identificar qué columna/hoja/fichero falla')
    print('    3. Corregir los Excel originales (o el contrato si la')
    print('       estructura ha cambiado legítimamente)')
    print('    4. Re-ejecutar este notebook')
    print('\n  ⚠️ NO ejecutar Fase 1 hasta que esta validación pase.')
elif resultado['todos_ok']:
    print('\n  ✓ VALIDACIÓN COMPLETADA con éxito — los 5 niveles pasados')
    print('    sin incidencias. Los Excel originales cumplen el contrato.')
    print()
    print('  Siguiente paso: continuar con')
    print('    notebooks/fase1_transformacion/f1_m01_reportes_raw.ipynb')
else:
    print('\n  ⚠️ VALIDACIÓN COMPLETADA con avisos — los niveles bloqueantes')
    print('    (N1-N3) han pasado, pero hay incidencias menores en N4 y/o N5.')
    print()
    print('  Siguiente paso:')
    print('    1. Revisar el HTML para ver los avisos')
    print('    2. Decidir si son aceptables o requieren acción')
    print('    3. Si son aceptables, continuar con F1')

print()
print('=' * 60)


CONCLUSIÓN

  ✓ VALIDACIÓN COMPLETADA con éxito — los 5 niveles pasados
    sin incidencias. Los Excel originales cumplen el contrato.

  Siguiente paso: continuar con
    notebooks/fase1_transformacion/f1_m01_reportes_raw.ipynb

